In [1]:
import gradio as gr
import torch
import torchvision.models as models
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import os

# ── 1. Class names (must match training order) ──────────────────
common_classes = sorted([
    'Apple_Scab_Leaf', 'Apple_leaf', 'Apple_rust_leaf',
    'Bell_pepper_leaf', 'Bell_pepper_leaf_spot',
    'Blueberry_leaf', 'Cherry_leaf', 'Corn_Common_rust_leaf',
    'Corn_Gray_leaf_spot', 'Corn_healthy_leaf', 'Corn_rust_leaf',
    'Grape_leaf', 'Grape_leaf_black_rot', 'Peach_leaf',
    'Potato_leaf', 'Potato_leaf_early_blight', 'Potato_leaf_late_blight',
    'Raspberry_leaf', 'Soybean_leaf', 'Squash_Powdery_mildew_leaf',
    'Strawberry_leaf', 'Tomato_Early_blight_leaf', 'Tomato_Septoria_leaf_spot',
    'Tomato_leaf', 'Tomato_leaf_bacterial_spot', 'Tomato_mold_leaf',
    'grape_leaf', 'grape_leaf_black_rot'
])

# ── 2. Load model ────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_model():
    model = models.efficientnet_b3(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(512, 27)
    )
    model.load_state_dict(torch.load('best_model.pth', map_location=device))
    model.eval()
    return model.to(device)

model = load_model()

# ── 3. Transforms ────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ── 4. Prediction function ───────────────────────────────────────
def predict_disease(image):
    # Preprocess
    img_tensor = transform(image).unsqueeze(0).to(device)

    # Predict
    with torch.no_grad():
        outputs = model(img_tensor)
        probs   = torch.softmax(outputs, dim=1)[0]

    # Top 3 predictions
    top3_probs, top3_indices = torch.topk(probs, 3)
    
    results = {}
    for prob, idx in zip(top3_probs, top3_indices):
        class_name = common_classes[idx.item()]
        # Make name readable
        readable = class_name.replace('_', ' ').title()
        results[readable] = float(prob)

    return results

# ── 5. Gradio Interface ──────────────────────────────────────────
demo = gr.Interface(
    fn=predict_disease,
    inputs=gr.Image(type="pil", label="Upload a Plant Leaf Image"),
    outputs=gr.Label(num_top_classes=3, label="Disease Prediction"),
    title="🌿 Crop Disease Detector",
    description="""
    ## AI-Powered Plant Disease Detection
    Upload a photo of a plant leaf and the model will identify the disease.
    
    **Supported plants:** Apple, Bell Pepper, Blueberry, Cherry, Corn, 
    Grape, Peach, Potato, Raspberry, Soybean, Squash, Strawberry, Tomato
    
    **Model:** EfficientNet-B3 | **Dataset:** PlantDoc (Real-world images)
    """,
    theme=gr.themes.Soft()
)

if __name__ == "__main__":
    demo.launch(share=True)  # share=True gives a public link!

C:\Users\lenovo\anaconda3\Lib\site-packages\gradio\interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://dc4c7de88750cb11a4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:
print(f"Number of classes: {len(common_classes)}")
for i, c in enumerate(common_classes):
    print(f"{i}: {c}")

Number of classes: 28
0: Apple_Scab_Leaf
1: Apple_leaf
2: Apple_rust_leaf
3: Bell_pepper_leaf
4: Bell_pepper_leaf_spot
5: Blueberry_leaf
6: Cherry_leaf
7: Corn_Common_rust_leaf
8: Corn_Gray_leaf_spot
9: Corn_healthy_leaf
10: Corn_rust_leaf
11: Grape_leaf
12: Grape_leaf_black_rot
13: Peach_leaf
14: Potato_leaf
15: Potato_leaf_early_blight
16: Potato_leaf_late_blight
17: Raspberry_leaf
18: Soybean_leaf
19: Squash_Powdery_mildew_leaf
20: Strawberry_leaf
21: Tomato_Early_blight_leaf
22: Tomato_Septoria_leaf_spot
23: Tomato_leaf
24: Tomato_leaf_bacterial_spot
25: Tomato_mold_leaf
26: grape_leaf
27: grape_leaf_black_rot


In [2]:
import gradio as gr
import torch
import torchvision.models as models
import torch.nn as nn
from torchvision import transforms
from PIL import Image

# ── 1. Class names ───────────────────────────────────────────────
common_classes = sorted([
    'Apple_Scab_Leaf', 'Apple_leaf', 'Apple_rust_leaf',
    'Bell_pepper_leaf', 'Bell_pepper_leaf_spot',
    'Blueberry_leaf', 'Cherry_leaf', 'Corn_Common_rust_leaf',
    'Corn_Gray_leaf_spot', 'Corn_healthy_leaf', 'Corn_rust_leaf',
    'Grape_leaf', 'Grape_leaf_black_rot', 'Peach_leaf',
    'Potato_leaf', 'Potato_leaf_early_blight', 'Potato_leaf_late_blight',
    'Raspberry_leaf', 'Soybean_leaf', 'Squash_Powdery_mildew_leaf',
    'Strawberry_leaf', 'Tomato_Early_blight_leaf', 'Tomato_Septoria_leaf_spot',
    'Tomato_leaf', 'Tomato_leaf_bacterial_spot', 'Tomato_mold_leaf',
    'grape_leaf_black_rot'
])

# ── 2. Load model ────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_model():
    model = models.efficientnet_b3(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(512, 27)
    )
    model.load_state_dict(torch.load('best_model.pth', map_location=device))
    model.eval()
    return model.to(device)

model = load_model()

# ── 3. Transforms ────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ── 4. Prediction function ───────────────────────────────────────
def predict_disease(image):
    if image is None:
        return {}, "Please upload an image first."

    img_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img_tensor)
        probs   = torch.softmax(outputs, dim=1)[0]

    top3_probs, top3_indices = torch.topk(probs, 3)

    results = {}
    top_class = ""
    for i, (prob, idx) in enumerate(zip(top3_probs, top3_indices)):
        class_name = common_classes[idx.item()]
        readable   = class_name.replace('_', ' ').title()
        results[readable] = float(prob)
        if i == 0:
            top_class = readable

    confidence = float(top3_probs[0]) * 100
    if confidence > 70:
        verdict = f"✅ High Confidence: **{top_class}** ({confidence:.1f}%)"
    elif confidence > 45:
        verdict = f"⚠️ Moderate Confidence: **{top_class}** ({confidence:.1f}%)"
    else:
        verdict = f"❓ Low Confidence: **{top_class}** ({confidence:.1f}%) — try a clearer image"

    return results, verdict


# ── 5. Custom Gradio Blocks UI ───────────────────────────────────
css = """
#title {text-align: center; font-size: 2.2em; font-weight: 800; 
        color: #2d6a4f; margin-bottom: 0px;}
#subtitle {text-align: center; color: #666; margin-bottom: 20px; font-size: 1.1em;}
#verdict {font-size: 1.2em; padding: 12px; border-radius: 10px; 
          background: #f0fdf4; border-left: 4px solid #2d6a4f;}
.gradio-container {max-width: 900px; margin: auto;}
#upload-box {border: 2px dashed #2d6a4f; border-radius: 16px;}
footer {display: none !important;}
"""

with gr.Blocks(css=css, theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 🌿 Crop Disease Detector", elem_id="title")
    gr.Markdown(
        "Upload a plant leaf photo → AI identifies the disease instantly",
        elem_id="subtitle"
    )

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(
                type="pil",
                label="📷 Upload Leaf Image",
                elem_id="upload-box",
                height=320
            )
            submit_btn = gr.Button(
                "🔍 Detect Disease",
                variant="primary",
                size="lg"
            )

        with gr.Column(scale=1):
            label_output  = gr.Label(
                num_top_classes=3,
                label="🧠 Top 3 Predictions"
            )
            verdict_output = gr.Markdown(
                label="Verdict",
                elem_id="verdict"
            )

    with gr.Accordion("ℹ️ Supported Plants & Diseases", open=False):
        gr.Markdown("""
        | Crop | Diseases Detected |
        |------|-------------------|
        | 🍎 Apple | Scab, Rust, Healthy |
        | 🌽 Corn | Common Rust, Gray Leaf Spot, Healthy |
        | 🍇 Grape | Black Rot, Healthy |
        | 🥔 Potato | Early Blight, Late Blight, Healthy |
        | 🍅 Tomato | Early Blight, Septoria Spot, Bacterial Spot, Mold |
        | 🫑 Bell Pepper | Leaf Spot, Healthy |
        | 🍓 Strawberry | Healthy |
        | + more | Blueberry, Cherry, Peach, Raspberry, Soybean, Squash |
        """)

    with gr.Accordion("📊 Model Details", open=False):
        gr.Markdown("""
        - **Architecture:** EfficientNet-B3 (Transfer Learning)
        - **Dataset:** PlantDoc — real-world field images
        - **Classes:** 27 disease categories across 13 crop species
        - **Training:** PyTorch | Augmentation: Flip, Rotate, ColorJitter
        - **Deployment:** Gradio + Hugging Face Spaces
        """)

    submit_btn.click(
        fn=predict_disease,
        inputs=image_input,
        outputs=[label_output, verdict_output]
    )

    gr.Markdown(
        "Built with ❤️ using PyTorch & EfficientNet-B3 | "
        "[GitHub](https://github.com) · [Dataset: PlantDoc](https://github.com/pratikkayal/PlantDoc-Dataset)",
        elem_id="subtitle"
    )

if __name__ == "__main__":
    demo.launch(share=True)

C:\Users\lenovo\AppData\Local\Temp\ipykernel_39544\3046205771.py:94: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=css, theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://025d71e39807ce0a13.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
